# S4-03: 건축공학 RAG 종합 실습
**KDS 설계기준 RAG / 건축 시방서 검색 / 구조공학 논문 검색**

## 학습 목표
- KDS 설계기준에 최적화된 조항 기반 RAG 시스템을 구축한다
- 건축 시방서 검색 시스템을 구현하고 현장 질의에 답변한다
- 구조공학 논문 하이브리드 검색 시스템을 구축한다

## 사전 준비
1. `.env` 파일에 API 키 설정
2. S4_01, S4_02에서 학습한 개념을 복습

> **모든 실습이 건축공학 도메인에 특화되어 있습니다.**

In [ ]:
# 패키지 설치
%pip install anthropic openai python-dotenv chromadb numpy rank-bm25

In [ ]:
# 환경 설정
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic
from openai import OpenAI
import chromadb
import numpy as np
import re
import json
from rank_bm25 import BM25Okapi

anthropic_client = Anthropic()
openai_client = OpenAI()
chroma_client = chromadb.Client()
model = "claude-sonnet-4-0"

def get_embeddings(texts: list[str]) -> list[list[float]]:
    """여러 텍스트의 임베딩을 한 번에 생성한다."""
    response = openai_client.embeddings.create(
        input=texts, model="text-embedding-3-small"
    )
    return [item.embedding for item in response.data]

def cosine_similarity(vec_a, vec_b) -> float:
    """코사인 유사도 계산"""
    a, b = np.array(vec_a), np.array(vec_b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def reciprocal_rank_fusion(rankings, k=60):
    """RRF로 여러 검색 결과를 결합한다."""
    rrf_scores = {}
    for ranking in rankings:
        for rank, (doc_id, _) in enumerate(ranking):
            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0) + 1.0 / (k + rank + 1)
    return sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)

class BM25Search:
    """BM25 기반 어휘 검색"""
    def __init__(self, documents):
        self.documents = documents
        self.tokenized = [re.findall(r'\w+', d.lower()) for d in documents]
        self.bm25 = BM25Okapi(self.tokenized)

    def search(self, query, top_k=3):
        tokens = re.findall(r'\w+', query.lower())
        scores = self.bm25.get_scores(tokens)
        ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)
        return ranked[:top_k]

print("환경 설정 완료")

---
## Exercise 1: KDS 구조기준 Contextual RAG 시스템

KDS 콘크리트구조 설계기준에 **Contextual Retrieval**을 적용한 고정밀 RAG 시스템을 구축하세요.

**요구사항:**
1. KDS 문서를 조항 번호 기준으로 청킹
2. 각 청크에 Claude(Haiku)를 사용하여 50-100 토큰의 맥락 설명 생성
3. 컨텍스추얼 청크로 하이브리드 검색 (시맨틱 + BM25) 수행
4. Claude에게 검색 결과를 전달하여 KDS 기준에 근거한 답변 생성
5. 일반 RAG와 Contextual RAG의 검색 결과 비교

**KDS 문서 데이터:**
아래 제공된 KDS 샘플 데이터를 사용하세요.

**테스트 질의:**
```python
queries = [
    "RC 기둥에 적용되는 강도감소계수는?",
    "내진설계범주 D에서 기둥 설계 시 주의사항은?",
    "500x500 기둥의 최대 축하중 강도를 구하라. fck=24MPa, fy=400MPa, 8-D25.",
]
```

**기대 출력:**
```
=== Contextual RAG 구축 ===
[kds-4.2.1] 맥락: 이 조항은 KDS 14 20 20의 4장에서 콘크리트 부재 설계의...
[kds-4.2.2] 맥락: 이 조항은 강도감소계수에 관한 기준으로, 인장/압축 지배...
...

=== 질의: RC 기둥에 적용되는 강도감소계수는? ===
[Contextual 검색] kds-4.2.2, kds-4.3.1, kds-4.2.1
[일반 검색]       kds-4.2.2, kds-5.1.1, kds-4.2.1  ← 5.1.1은 보 관련!

A: KDS 4.2.2에 따르면, 기둥(압축지배 단면)의 강도감소계수는...
```

In [ ]:
# KDS 설계기준 문서
kds_full_doc = """KDS 14 20 20 콘크리트구조 부재 설계기준

4장 기둥 설계
4.2.1 일반 사항: 콘크리트구조 부재의 설계는 극한강도설계법에 따른다. 모든 부재는 소요강도 이상의 설계강도를 가져야 한다. 설계강도 = 강도감소계수(phi) x 공칭강도(Rn).
4.2.2 강도감소계수: 인장지배 단면의 강도감소계수는 0.85로 한다. 압축지배 단면의 경우 나선철근 부재는 0.70, 기타 부재는 0.65로 한다.
4.3.1 축하중을 받는 부재: 축방향 압축력을 받는 부재의 공칭강도 Pn = 0.80 * [0.85 * fck * (Ag - Ast) + fy * Ast]. 여기서 Ag는 전체 단면적, Ast는 철근 단면적이다.
4.4.1 최소 철근비: 기둥의 종방향 철근비는 전체 단면적의 1% 이상, 8% 이하로 한다. 최소 4개의 종방향 철근을 배치해야 한다.
4.4.2 띠철근 간격: 띠철근 간격은 다음 중 작은 값 이하로 한다. (1) 종방향 철근 지름의 16배 (2) 띠철근 지름의 48배 (3) 기둥 단면의 최소 치수.

5장 보 설계
5.1.1 보의 휨 설계: 보의 공칭 휨강도 Mn = As * fy * (d - a/2). 여기서 a = As * fy / (0.85 * fck * b). 보의 인장 철근비는 균형 철근비의 75% 이하로 제한한다.
5.2.1 보의 전단 설계: 콘크리트가 부담하는 전단강도 Vc = (1/6) * sqrt(fck) * b * d. 전단철근이 부담하는 전단강도 Vs = Av * fy * d / s.

6장 내진설계
6.1.1 내진설계 일반: 내진설계범주 D 이상의 구조물에서 특수 모멘트골조를 사용하는 경우, 기둥의 강도는 보 강도의 1.2배 이상이어야 한다 (강기둥-약보 원칙)."""

kds_documents = [
    {"id": "kds-4.2.1", "text": "4.2.1 일반 사항: 콘크리트구조 부재의 설계는 극한강도설계법에 따른다. 모든 부재는 소요강도 이상의 설계강도를 가져야 한다. 설계강도 = 강도감소계수(phi) x 공칭강도(Rn).", "metadata": {"clause": "4.2.1", "chapter": "4", "topic": "일반"}},
    {"id": "kds-4.2.2", "text": "4.2.2 강도감소계수: 인장지배 단면의 강도감소계수는 0.85로 한다. 압축지배 단면의 경우 나선철근 부재는 0.70, 기타 부재는 0.65로 한다.", "metadata": {"clause": "4.2.2", "chapter": "4", "topic": "강도감소계수"}},
    {"id": "kds-4.3.1", "text": "4.3.1 축하중을 받는 부재: 축방향 압축력을 받는 부재의 공칭강도 Pn = 0.80 * [0.85 * fck * (Ag - Ast) + fy * Ast]. 여기서 Ag는 전체 단면적, Ast는 철근 단면적이다.", "metadata": {"clause": "4.3.1", "chapter": "4", "topic": "축하중"}},
    {"id": "kds-4.4.1", "text": "4.4.1 최소 철근비: 기둥의 종방향 철근비는 전체 단면적의 1% 이상, 8% 이하로 한다. 최소 4개의 종방향 철근을 배치해야 한다.", "metadata": {"clause": "4.4.1", "chapter": "4", "topic": "철근비"}},
    {"id": "kds-4.4.2", "text": "4.4.2 띠철근 간격: 띠철근 간격은 다음 중 작은 값 이하로 한다. (1) 종방향 철근 지름의 16배 (2) 띠철근 지름의 48배 (3) 기둥 단면의 최소 치수.", "metadata": {"clause": "4.4.2", "chapter": "4", "topic": "띠철근"}},
    {"id": "kds-5.1.1", "text": "5.1.1 보의 휨 설계: 보의 공칭 휨강도 Mn = As * fy * (d - a/2). 여기서 a = As * fy / (0.85 * fck * b). 보의 인장 철근비는 균형 철근비의 75% 이하로 제한한다.", "metadata": {"clause": "5.1.1", "chapter": "5", "topic": "보 휨설계"}},
    {"id": "kds-5.2.1", "text": "5.2.1 보의 전단 설계: 콘크리트가 부담하는 전단강도 Vc = (1/6) * sqrt(fck) * b * d. 전단철근이 부담하는 전단강도 Vs = Av * fy * d / s.", "metadata": {"clause": "5.2.1", "chapter": "5", "topic": "보 전단설계"}},
    {"id": "kds-6.1.1", "text": "6.1.1 내진설계 일반: 내진설계범주 D 이상의 구조물에서 특수 모멘트골조를 사용하는 경우, 기둥의 강도는 보 강도의 1.2배 이상이어야 한다 (강기둥-약보 원칙).", "metadata": {"clause": "6.1.1", "chapter": "6", "topic": "내진설계"}},
]

queries = [
    "RC 기둥에 적용되는 강도감소계수는?",
    "내진설계범주 D에서 기둥 설계 시 주의사항은?",
    "500x500 기둥의 최대 축하중 강도를 구하라. fck=24MPa, fy=400MPa, 8-D25.",
]

# TODO: Contextual RAG 시스템을 구현하세요

In [ ]:
# ===== 정답 =====

class ContextualKDSRAG:
    """KDS 구조기준 Contextual RAG 시스템"""

    def __init__(self, full_doc: str, documents: list[dict]):
        self.full_doc = full_doc
        self.documents = documents
        self.doc_map = {d["id"]: d for d in documents}
        self.doc_texts = [d["text"] for d in documents]

        # 1. 맥락 생성
        print("=== Contextual RAG 구축 ===")
        self.contextual_docs = []
        for doc in documents:
            context = self._generate_context(doc["text"])
            ctx_doc = {
                **doc,
                "context": context,
                "contextual_text": f"{context}\n\n{doc['text']}"
            }
            self.contextual_docs.append(ctx_doc)
            print(f"[{doc['id']}] 맥락: {context[:60]}...")

        # 2. 컨텍스추얼 인덱스 구축
        ctx_texts = [d["contextual_text"] for d in self.contextual_docs]
        ctx_embeddings = get_embeddings(ctx_texts)
        self.ctx_collection = chroma_client.get_or_create_collection(
            name="kds_contextual_ex1"
        )
        self.ctx_collection.add(
            ids=[d["id"] for d in documents],
            documents=ctx_texts,
            metadatas=[d["metadata"] for d in documents],
            embeddings=ctx_embeddings
        )
        self.ctx_bm25 = BM25Search(ctx_texts)

        # 3. 일반 인덱스 (비교용)
        plain_embeddings = get_embeddings(self.doc_texts)
        self.plain_collection = chroma_client.get_or_create_collection(
            name="kds_plain_ex1"
        )
        self.plain_collection.add(
            ids=[d["id"] for d in documents],
            documents=self.doc_texts,
            metadatas=[d["metadata"] for d in documents],
            embeddings=plain_embeddings
        )

        print(f"\n인덱스 구축 완료: {len(documents)}개 문서")

    def _generate_context(self, chunk_text: str) -> str:
        """Claude Haiku로 청크 맥락을 생성한다."""
        response = anthropic_client.messages.create(
            model="claude-haiku-4-5",
            max_tokens=150,
            temperature=0.0,
            messages=[{
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": f"<document>\n{self.full_doc}\n</document>",
                        "cache_control": {"type": "ephemeral"}
                    },
                    {
                        "type": "text",
                        "text": f"\n위 문서에서 다음 청크의 위치와 맥락을 50-100 토큰으로 설명하세요.\n\n<chunk>\n{chunk_text}\n</chunk>\n\n맥락 설명:"
                    }
                ]
            }]
        )
        return response.content[0].text.strip()

    def search_contextual(self, query: str, top_k: int = 3) -> list[str]:
        """Contextual 하이브리드 검색"""
        q_emb = get_embeddings([query])[0]
        sem = self.ctx_collection.query(
            query_embeddings=[q_emb], n_results=top_k, include=["distances"]
        )
        sem_ranking = [(sem["ids"][0][i], 1.0 - sem["distances"][0][i])
                       for i in range(len(sem["ids"][0]))]
        bm25_res = self.ctx_bm25.search(query, top_k)
        bm25_ranking = [(self.documents[idx]["id"], sc) for idx, sc in bm25_res]
        fused = reciprocal_rank_fusion([sem_ranking, bm25_ranking])
        return [doc_id for doc_id, _ in fused[:top_k]]

    def search_plain(self, query: str, top_k: int = 3) -> list[str]:
        """일반 시맨틱 검색 (비교용)"""
        q_emb = get_embeddings([query])[0]
        sem = self.plain_collection.query(
            query_embeddings=[q_emb], n_results=top_k, include=["distances"]
        )
        return sem["ids"][0][:top_k]

    def query(self, question: str, top_k: int = 3) -> str:
        """Contextual RAG 질의"""
        ctx_ids = self.search_contextual(question, top_k)
        plain_ids = self.search_plain(question, top_k)

        print(f"[Contextual 검색] {', '.join(ctx_ids)}")
        print(f"[일반 검색]       {', '.join(plain_ids)}")

        # Contextual 검색 결과로 프롬프트 조립
        context = "\n\n".join(
            f"[KDS {self.doc_map[did]['metadata']['clause']}] {self.doc_map[did]['text']}"
            for did in ctx_ids if did in self.doc_map
        )

        prompt = f"""당신은 KDS 콘크리트구조 설계기준 전문가입니다.

검색된 KDS 조항:
{context}

규칙:
1. 반드시 위 조항을 근거로 답변하세요
2. 조항 번호를 인용하세요 (예: KDS 4.2.2에 따르면...)
3. 계산이 필요하면 단계별로 보여주세요

질문: {question}"""

        response = anthropic_client.messages.create(
            model=model, max_tokens=1500, temperature=0.1,
            messages=[{"role": "user", "content": prompt}]
        )
        return response.content[0].text

# Contextual RAG 구축 및 실행
ctx_rag = ContextualKDSRAG(kds_full_doc, kds_documents)

for q in queries:
    print(f"\n{'='*60}")
    print(f"Q: {q}")
    print(f"{'='*60}")
    answer = ctx_rag.query(q)
    print(f"\nA: {answer}")

def verify_ex1():
    # Contextual 검색이 기둥 관련 질의에서 보 관련 문서를 제외하는지 확인
    ctx_ids = ctx_rag.search_contextual("RC 기둥의 강도감소계수", top_k=3)
    # 기둥 관련 질의에서 4장 문서가 우선되어야 함
    chapter_4_count = sum(1 for did in ctx_ids if did in ctx_rag.doc_map and ctx_rag.doc_map[did]["metadata"]["chapter"] == "4")
    assert chapter_4_count >= 2, "기둥 질의에서 4장 문서가 2개 이상 포함되어야 합니다"
    print("\nExercise 1 통과!")

verify_ex1()

---
## Exercise 2: 건축 시방서 검색 시스템

건축 공사 시방서를 RAG로 구축하여, 시공 현장에서 발생하는 질의에 즉시 답변하는 시스템을 구현하세요.

**요구사항:**
1. 제공된 시방서 문서를 ChromaDB에 저장
2. BM25 + 시맨틱 하이브리드 검색 구현
3. 현장 엔지니어의 5가지 질의에 답변
4. 답변에 시방서 섹션 정보 포함

**시방서 문서:**
아래 제공된 건축 시방서 샘플 데이터를 사용하세요.

**테스트 질의:**
```python
queries = [
    "기초 콘크리트의 피복 두께는 얼마인가요?",
    "한겨울에 콘크리트 타설 시 주의사항은?",
    "콘크리트 배합의 물-시멘트비 기준은?",
    "양생 기간은 최소 며칠인가요?",
    "철근 이음 겹침 길이는?",
]
```

In [ ]:
# 건축 시방서 문서
spec_documents = [
    {"id": "spec-mix", "text": "콘크리트 배합: 설계기준강도 24MPa 이상의 콘크리트를 사용한다. 물-시멘트비는 0.55 이하로 하며, 슬럼프는 120-150mm, 공기량은 4.5+-1.5%로 한다. AE제를 사용하여 동결융해 저항성을 확보한다.", "metadata": {"section": "콘크리트", "topic": "배합", "page": 12}},
    {"id": "spec-cover", "text": "피복 두께: 기둥과 보의 최소 피복 두께는 40mm로 한다. 기초의 피복 두께는 흙에 접하는 면 80mm, 그 외 면 60mm로 한다. 슬래브의 피복 두께는 20mm (옥내) 또는 40mm (옥외)로 한다.", "metadata": {"section": "철근", "topic": "피복두께", "page": 18}},
    {"id": "spec-curing", "text": "양생: 콘크리트 타설 후 최소 5일간 습윤양생을 실시한다. 외기온도가 5도 이하인 경우 보온양생을 병행하며, 35도 이상인 경우 서중 콘크리트 대책을 적용한다. 양생 중 충격과 진동을 피한다.", "metadata": {"section": "콘크리트", "topic": "양생", "page": 15}},
    {"id": "spec-rebar-splice", "text": "철근 이음: 겹침이음의 길이는 40d 이상으로 한다 (d: 철근 지름). D25 이상의 철근은 기계식 이음 또는 용접이음을 사용한다. 같은 단면에서 이음은 전체 철근의 50% 이하로 한다.", "metadata": {"section": "철근", "topic": "이음", "page": 20}},
    {"id": "spec-formwork", "text": "거푸집: 기둥 거푸집의 수직도 허용오차는 H/500 이내로 한다. 보 거푸집의 존치 기간은 콘크리트 강도가 설계기준강도의 2/3 이상 도달할 때까지로 한다. 거푸집 제거 전 강도 시험을 실시한다.", "metadata": {"section": "거푸집", "topic": "설치기준", "page": 22}},
    {"id": "spec-cold", "text": "한중 콘크리트: 일평균 기온이 4도 이하인 경우 한중 콘크리트 대책을 적용한다. 타설 시 콘크리트 온도는 10-20도를 유지한다. 보온 양생 기간은 압축강도 5MPa 도달 시까지로 한다.", "metadata": {"section": "콘크리트", "topic": "한중", "page": 16}},
    {"id": "spec-placing", "text": "콘크리트 타설: 자유낙하 높이는 1.5m 이하로 한다. 1층 타설 높이는 40-50cm 이내로 하고, 바이브레이터로 다짐한다. 이어치기 시간 간격은 외기 25도 이상에서 1.5시간, 25도 미만에서 2.5시간 이내로 한다.", "metadata": {"section": "콘크리트", "topic": "타설", "page": 14}},
    {"id": "spec-inspection", "text": "품질 검사: 콘크리트 압축강도 시험은 타설 일마다 1회 이상 실시한다. 시험체는 3개를 1조로 하며, 28일 강도가 설계기준강도 이상이어야 한다. 슬럼프 시험은 운반차마다 실시한다.", "metadata": {"section": "품질관리", "topic": "검사", "page": 25}},
]

queries = [
    "기초 콘크리트의 피복 두께는 얼마인가요?",
    "한겨울에 콘크리트 타설 시 주의사항은?",
    "콘크리트 배합의 물-시멘트비 기준은?",
    "양생 기간은 최소 며칠인가요?",
    "철근 이음 겹침 길이는?",
]

# TODO: 시방서 검색 시스템을 구현하세요

In [ ]:
# ===== 정답 =====

class SpecificationRAG:
    """건축 시방서 하이브리드 RAG 시스템"""

    def __init__(self, documents: list[dict]):
        self.documents = documents
        self.doc_map = {d["id"]: d for d in documents}
        self.doc_texts = [d["text"] for d in documents]

        # 시맨틱 인덱스
        embeddings = get_embeddings(self.doc_texts)
        self.collection = chroma_client.get_or_create_collection(
            name="construction_spec_ex2"
        )
        self.collection.add(
            ids=[d["id"] for d in documents],
            documents=self.doc_texts,
            metadatas=[d["metadata"] for d in documents],
            embeddings=embeddings
        )

        # BM25 인덱스
        self.bm25 = BM25Search(self.doc_texts)
        print(f"시방서 검색 시스템 초기화: {len(documents)}개 조항")

    def search(self, query: str, top_k: int = 3) -> list[dict]:
        """하이브리드 검색"""
        # 시맨틱
        q_emb = get_embeddings([query])[0]
        sem = self.collection.query(
            query_embeddings=[q_emb], n_results=top_k,
            include=["distances"]
        )
        sem_ranking = [(sem["ids"][0][i], 1.0 - sem["distances"][0][i])
                       for i in range(len(sem["ids"][0]))]

        # BM25
        bm25_res = self.bm25.search(query, top_k)
        bm25_ranking = [(self.documents[idx]["id"], sc) for idx, sc in bm25_res]

        # RRF
        fused = reciprocal_rank_fusion([sem_ranking, bm25_ranking])

        return [
            self.doc_map[did] for did, _ in fused[:top_k]
            if did in self.doc_map
        ]

    def answer(self, question: str, top_k: int = 2) -> str:
        """검색 + Claude 답변 생성"""
        results = self.search(question, top_k)

        context = "\n\n".join(
            f"[{r['metadata']['section']}/{r['metadata']['topic']} (p.{r['metadata']['page']})] {r['text']}"
            for r in results
        )

        prompt = f"""당신은 건축 시공 전문가입니다.

검색된 시방서 조항:
{context}

현장 질의에 명확하고 간결하게 답변하세요. 시방서 출처를 명시하세요.

질문: {question}"""

        response = anthropic_client.messages.create(
            model=model, max_tokens=512, temperature=0.1,
            messages=[{"role": "user", "content": prompt}]
        )
        return response.content[0].text

# 시스템 구축 및 테스트
spec_rag = SpecificationRAG(spec_documents)

for q in queries:
    print(f"\nQ: {q}")
    results = spec_rag.search(q, top_k=2)
    print(f"  검색: {', '.join(r['id'] for r in results)}")
    answer = spec_rag.answer(q, top_k=2)
    print(f"  A: {answer[:200]}...")
    print(f"  {'-'*40}")

def verify_ex2():
    # 피복 두께 질의
    results = spec_rag.search("기초 피복 두께", top_k=1)
    assert results[0]["id"] == "spec-cover", "피복 두께 질의에 spec-cover가 1위여야 합니다"

    # 한중 콘크리트 질의
    results = spec_rag.search("한겨울 콘크리트 타설", top_k=2)
    ids = [r["id"] for r in results]
    assert "spec-cold" in ids, "한중 콘크리트 질의에 spec-cold가 포함되어야 합니다"

    # 이음 길이 질의
    results = spec_rag.search("철근 겹침이음 길이", top_k=1)
    assert results[0]["id"] == "spec-rebar-splice", "이음 질의에 spec-rebar-splice가 1위여야 합니다"

    print("\nExercise 2 통과!")

verify_ex2()

---
## Exercise 3: 구조공학 논문 하이브리드 검색

연구실 논문 초록 데이터를 하이브리드 RAG로 검색하여 문헌 조사를 지원하는 시스템을 구축하세요.

**요구사항:**
1. 구조공학 논문 초록 10개를 ChromaDB + BM25에 인덱싱
2. 하이브리드 검색 (시맨틱 + BM25 + RRF) 구현
3. 검색 결과를 Claude에게 전달하여 문헌 리뷰 초안 생성
4. 검색 품질 평가: Precision@3 계산

**검색 품질 평가:**
```python
# 정답 레이블 (각 질의에 대한 관련 논문 ID)
ground_truth = {
    "coupling beam seismic ductility": ["paper-001", "paper-004"],
    "deep learning structural prediction": ["paper-002", "paper-005"],
    "high strength concrete column": ["paper-003"],
}
```

**기대 출력:**
```
=== 문헌 검색: coupling beam seismic ductility ===
  [1] paper-001 (RRF: 0.0xxx) — ACI Structural (2023)
  [2] paper-004 (RRF: 0.0xxx) — Earthquake Engineering (2023)
  ...
  Precision@3: 0.67

=== 전체 검색 품질 ===
  평균 Precision@3: 0.xx
```

In [ ]:
# 구조공학 논문 초록 데이터
papers = [
    {"id": "paper-001", "text": "This study investigates the seismic performance of reinforced concrete coupling beams with diagonal reinforcement. Experimental results show that the diagonal reinforcement ratio of 1.2% provides optimal ductility with a displacement ductility factor of 5.2.", "metadata": {"year": 2023, "topic": "coupling beam", "journal": "ACI Structural", "keywords": "coupling beam, diagonal reinforcement, ductility"}},
    {"id": "paper-002", "text": "A deep learning model based on convolutional neural networks is proposed for predicting the shear strength of reinforced concrete beams. The model achieves R-squared of 0.96 on the test dataset with 1200 samples.", "metadata": {"year": 2024, "topic": "deep learning", "journal": "Engineering Structures", "keywords": "deep learning, CNN, shear strength, prediction"}},
    {"id": "paper-003", "text": "Experimental investigation of high-strength concrete columns under combined axial load and biaxial bending. The test results indicate that the ACI 318 interaction diagram is conservative for fck > 60MPa.", "metadata": {"year": 2024, "topic": "HSC column", "journal": "ASCE Structural", "keywords": "high strength concrete, column, biaxial bending"}},
    {"id": "paper-004", "text": "Non-linear finite element analysis of reinforced concrete shear walls subjected to reversed cyclic loading. The study proposes a modified damage model that captures strength degradation and pinching behavior.", "metadata": {"year": 2023, "topic": "shear wall", "journal": "Earthquake Engineering", "keywords": "shear wall, FEM, cyclic loading, damage model"}},
    {"id": "paper-005", "text": "Machine learning-based rapid seismic damage assessment of buildings using structural health monitoring data. Random Forest classifier achieves 94% accuracy in identifying damage states.", "metadata": {"year": 2024, "topic": "ML damage", "journal": "SDEE", "keywords": "machine learning, damage assessment, SHM"}},
    {"id": "paper-006", "text": "Seismic performance evaluation of precast concrete moment frames with emulative connections. The hybrid connection system achieves comparable ductility to cast-in-place construction.", "metadata": {"year": 2024, "topic": "precast", "journal": "PCI Journal", "keywords": "precast, moment frame, seismic, emulative connection"}},
    {"id": "paper-007", "text": "Fiber-reinforced polymer retrofit of deficient reinforced concrete columns for seismic strengthening. CFRP wrapping increases the axial load capacity by 35% and ductility by 60%.", "metadata": {"year": 2023, "topic": "FRP retrofit", "journal": "Composites B", "keywords": "FRP, CFRP, column retrofit, seismic strengthening"}},
    {"id": "paper-008", "text": "Parametric study on the punching shear strength of flat slabs using finite element analysis. The study evaluates the effect of column aspect ratio, slab thickness, and flexural reinforcement ratio.", "metadata": {"year": 2024, "topic": "punching shear", "journal": "Structural Concrete", "keywords": "punching shear, flat slab, FEM, parametric study"}},
    {"id": "paper-009", "text": "Development of a physics-informed neural network for predicting the load-displacement behavior of reinforced concrete beam-column joints under seismic loading.", "metadata": {"year": 2024, "topic": "PINN", "journal": "CMAME", "keywords": "PINN, beam-column joint, seismic, physics-informed"}},
    {"id": "paper-010", "text": "Experimental study on coupling beams with ultra-high performance concrete subjected to reversed cyclic loading. UHPC coupling beams show 40% higher shear capacity compared to conventional RC coupling beams.", "metadata": {"year": 2024, "topic": "UHPC coupling beam", "journal": "ACI Structural", "keywords": "UHPC, coupling beam, cyclic loading, shear capacity"}},
]

ground_truth = {
    "coupling beam seismic ductility": ["paper-001", "paper-010", "paper-004"],
    "deep learning structural prediction": ["paper-002", "paper-005", "paper-009"],
    "high strength concrete column biaxial": ["paper-003", "paper-007"],
}

# TODO: 논문 하이브리드 검색 시스템과 검색 품질 평가를 구현하세요

In [ ]:
# ===== 정답 =====

class PaperSearchRAG:
    """구조공학 논문 하이브리드 검색 시스템"""

    def __init__(self, papers: list[dict]):
        self.papers = papers
        self.paper_map = {p["id"]: p for p in papers}
        self.paper_texts = [p["text"] for p in papers]

        # 시맨틱 인덱스
        embeddings = get_embeddings(self.paper_texts)
        self.collection = chroma_client.get_or_create_collection(
            name="struct_papers_ex3"
        )
        self.collection.add(
            ids=[p["id"] for p in papers],
            documents=self.paper_texts,
            metadatas=[{k: str(v) for k, v in p["metadata"].items()} for p in papers],
            embeddings=embeddings
        )

        # BM25 인덱스
        self.bm25 = BM25Search(self.paper_texts)
        print(f"논문 검색 시스템 초기화: {len(papers)}편")

    def search(self, query: str, top_k: int = 5) -> list[dict]:
        """하이브리드 검색"""
        # 시맨틱
        q_emb = get_embeddings([query])[0]
        sem = self.collection.query(
            query_embeddings=[q_emb], n_results=top_k, include=["distances"]
        )
        sem_ranking = [(sem["ids"][0][i], 1.0 - sem["distances"][0][i])
                       for i in range(len(sem["ids"][0]))]

        # BM25
        bm25_res = self.bm25.search(query, top_k)
        bm25_ranking = [(self.papers[idx]["id"], sc) for idx, sc in bm25_res]

        # RRF
        fused = reciprocal_rank_fusion([sem_ranking, bm25_ranking])

        return [
            {**self.paper_map[did], "rrf_score": score}
            for did, score in fused[:top_k]
            if did in self.paper_map
        ]

    def generate_review(self, topic: str, top_k: int = 3) -> str:
        """검색 결과를 기반으로 문헌 리뷰 초안 생성"""
        results = self.search(topic, top_k)

        papers_text = "\n\n".join(
            f"[{r['metadata']['journal']} ({r['metadata']['year']})] {r['text']}"
            for r in results
        )

        prompt = f"""당신은 구조공학 연구자입니다.

다음 논문들을 기반으로 '{topic}'에 대한 간결한 문헌 리뷰 (3-5문장)를 작성하세요.
각 논문의 핵심 기여를 언급하세요.

관련 논문:
{papers_text}"""

        response = anthropic_client.messages.create(
            model=model, max_tokens=512, temperature=0.3,
            messages=[{"role": "user", "content": prompt}]
        )
        return response.content[0].text

def precision_at_k(retrieved: list[str], relevant: list[str], k: int) -> float:
    """Precision@K: 상위 K개 검색 결과 중 관련 문서 비율"""
    top_k = retrieved[:k]
    relevant_in_top_k = sum(1 for doc_id in top_k if doc_id in relevant)
    return relevant_in_top_k / k

# 시스템 구축
paper_rag = PaperSearchRAG(papers)

# 검색 및 평가
precisions = []
for query, relevant_ids in ground_truth.items():
    results = paper_rag.search(query, top_k=5)
    retrieved_ids = [r["id"] for r in results]

    p_at_3 = precision_at_k(retrieved_ids, relevant_ids, k=3)
    precisions.append(p_at_3)

    print(f"\n=== 문헌 검색: {query} ===")
    for i, r in enumerate(results[:3]):
        marker = "O" if r["id"] in relevant_ids else "X"
        print(f"  [{marker}] {r['id']} (RRF: {r['rrf_score']:.6f}) — {r['metadata']['journal']} ({r['metadata']['year']})")
    print(f"  Precision@3: {p_at_3:.2f}")

mean_precision = np.mean(precisions)
print(f"\n=== 전체 검색 품질 ===")
print(f"  평균 Precision@3: {mean_precision:.2f}")

# 문헌 리뷰 생성 (첫 번째 주제)
print(f"\n{'='*60}")
print(f"=== 문헌 리뷰 생성: coupling beam seismic performance ===")
print(f"{'='*60}")
review = paper_rag.generate_review("coupling beam seismic performance", top_k=3)
print(review)

def verify_ex3():
    # coupling beam 검색 - paper-001이 상위에 있어야 함
    results = paper_rag.search("coupling beam ductility", top_k=3)
    top_ids = [r["id"] for r in results]
    assert "paper-001" in top_ids, "coupling beam 질의에 paper-001이 Top-3에 포함되어야 합니다"

    # precision 함수 기본 테스트
    assert precision_at_k(["A", "B", "C"], ["A", "C"], k=3) == 2/3, "Precision@3 계산 오류"
    assert precision_at_k(["A", "B", "C"], ["D"], k=3) == 0.0, "Precision@3 계산 오류"

    # 평균 precision이 0보다 커야 함
    assert mean_precision > 0, "평균 Precision이 0보다 커야 합니다"

    print("\nExercise 3 통과!")

verify_ex3()